<a href="https://colab.research.google.com/github/mdimssptr/UPRAK_G211220104-FUZZYLOGIC/blob/main/G211220104_UPRAKFUUZYLOGICPREDIKJAGUNG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

# =====================================================
# KASUS B: ANFIS - PREDIKSI PRODUKSI JAGUNG
# =====================================================

class ANFIS:
    def __init__(self):
        # Parameter Sugeno untuk setiap rule
        self.params = {
            'R1': {'p': 0.0692, 'q': 0.0461, 'r': 0.00231},
            'R2': {'p': 0.0698, 'q': 0.0390, 'r': 0.00139},
            'R3': {'p': 0.0585, 'q': 0.0183, 'r': 0.00084},
            'R4': {'p': 0.0505, 'q': 0.0318, 'r': 0.00115}
        }

        # Parameter fungsi keanggotaan Gaussian
        self.mf_params = {
            'A1': {'c': 40, 'sigma': 15},
            'A2': {'c': 70, 'sigma': 15},
            'B1': {'c': 20, 'sigma': 8},
            'B2': {'c': 30, 'sigma': 8}
        }

    def gaussian_mf(self, x, c, sigma):
        """Fungsi keanggotaan Gaussian"""
        return np.exp(-((x - c)**2) / (2 * sigma**2))

    def layer1_fuzzifikasi(self, x1, x2):
        """Layer 1: Fuzzifikasi dengan fungsi Gaussian"""
        mu_A1 = self.gaussian_mf(x1, self.mf_params['A1']['c'], self.mf_params['A1']['sigma'])
        mu_A2 = self.gaussian_mf(x1, self.mf_params['A2']['c'], self.mf_params['A2']['sigma'])
        mu_B1 = self.gaussian_mf(x2, self.mf_params['B1']['c'], self.mf_params['B1']['sigma'])
        mu_B2 = self.gaussian_mf(x2, self.mf_params['B2']['c'], self.mf_params['B2']['sigma'])

        return mu_A1, mu_A2, mu_B1, mu_B2

    def layer2_firing_strength(self, mu_A1, mu_A2, mu_B1, mu_B2):
        """Layer 2: Hitung firing strength (AND operation)"""
        w1 = mu_A1 * mu_B1
        w2 = mu_A1 * mu_B2
        w3 = mu_A2 * mu_B1
        w4 = mu_A2 * mu_B2

        return w1, w2, w3, w4

    def layer3_normalisasi(self, w1, w2, w3, w4):
        """Layer 3: Normalisasi firing strength"""
        total = w1 + w2 + w3 + w4

        if total == 0:
            return 0, 0, 0, 0

        w_bar1 = w1 / total
        w_bar2 = w2 / total
        w_bar3 = w3 / total
        w_bar4 = w4 / total

        return w_bar1, w_bar2, w_bar3, w_bar4

    def layer4_output_rule(self, x1, x2):
        """Layer 4: Hitung output tiap rule (Sugeno)"""
        f1 = self.params['R1']['p']*x1 + self.params['R1']['q']*x2 + self.params['R1']['r']
        f2 = self.params['R2']['p']*x1 + self.params['R2']['q']*x2 + self.params['R2']['r']
        f3 = self.params['R3']['p']*x1 + self.params['R3']['q']*x2 + self.params['R3']['r']
        f4 = self.params['R4']['p']*x1 + self.params['R4']['q']*x2 + self.params['R4']['r']

        return f1, f2, f3, f4

    def layer5_agregasi(self, w_bar1, w_bar2, w_bar3, w_bar4, f1, f2, f3, f4):
        """Layer 5: Agregasi (weighted average)"""
        output = w_bar1*f1 + w_bar2*f2 + w_bar3*f3 + w_bar4*f4
        return output

    def prediksi(self, x1, x2):
        """Proses prediksi lengkap menggunakan ANFIS"""
        print(f"\n📊 DATA INPUT:")
        print(f"  Soil Moisture (x₁): {x1}%")
        print(f"  Soil pH/Temp (x₂): {x2}")

        # Layer 1
        print("\n🔍 LAYER 1: FUZZIFIKASI (Gaussian)")
        mu_A1, mu_A2, mu_B1, mu_B2 = self.layer1_fuzzifikasi(x1, x2)
        print(f"  μ_A1({x1}) = exp(-((60-40)²)/(2×15²)) = {mu_A1:.4f}")
        print(f"  μ_A2({x1}) = exp(-((60-70)²)/(2×15²)) = {mu_A2:.4f}")
        print(f"  μ_B1({x2}) = exp(-((26-20)²)/(2×8²)) = {mu_B1:.4f}")
        print(f"  μ_B2({x2}) = exp(-((26-30)²)/(2×8²)) = {mu_B2:.4f}")

        # Layer 2
        print("\n⚙️  LAYER 2: FIRING STRENGTH")
        w1, w2, w3, w4 = self.layer2_firing_strength(mu_A1, mu_A2, mu_B1, mu_B2)
        print(f"  w₁ = μ_A1 × μ_B1 = {mu_A1:.4f} × {mu_B1:.4f} = {w1:.4f}")
        print(f"  w₂ = μ_A1 × μ_B2 = {mu_A1:.4f} × {mu_B2:.4f} = {w2:.4f}")
        print(f"  w₃ = μ_A2 × μ_B1 = {mu_A2:.4f} × {mu_B1:.4f} = {w3:.4f}")
        print(f"  w₄ = μ_A2 × μ_B2 = {mu_A2:.4f} × {mu_B2:.4f} = {w4:.4f}")
        print(f"  Total = {w1+w2+w3+w4:.4f}")

        # Layer 3
        print("\n🔄 LAYER 3: NORMALISASI")
        w_bar1, w_bar2, w_bar3, w_bar4 = self.layer3_normalisasi(w1, w2, w3, w4)
        print(f"  w̄₁ = {w1:.4f} / {w1+w2+w3+w4:.4f} = {w_bar1:.4f}")
        print(f"  w̄₂ = {w2:.4f} / {w1+w2+w3+w4:.4f} = {w_bar2:.4f}")
        print(f"  w̄₃ = {w3:.4f} / {w1+w2+w3+w4:.4f} = {w_bar3:.4f}")
        print(f"  w̄₄ = {w4:.4f} / {w1+w2+w3+w4:.4f} = {w_bar4:.4f}")
        print(f"  Verifikasi: Σw̄ = {w_bar1+w_bar2+w_bar3+w_bar4:.4f} ≈ 1.0 ✓")

        # Layer 4
        print("\n📐 LAYER 4: OUTPUT TIAP RULE (Sugeno Orde-1)")
        f1, f2, f3, f4 = self.layer4_output_rule(x1, x2)
        print(f"  f₁ = {self.params['R1']['p']}×{x1} + {self.params['R1']['q']}×{x2} + {self.params['R1']['r']}")
        print(f"     = {self.params['R1']['p']*x1:.4f} + {self.params['R1']['q']*x2:.4f} + {self.params['R1']['r']}")
        print(f"     = {f1:.4f}")

        print(f"  f₂ = {self.params['R2']['p']}×{x1} + {self.params['R2']['q']}×{x2} + {self.params['R2']['r']}")
        print(f"     = {self.params['R2']['p']*x1:.4f} + {self.params['R2']['q']*x2:.4f} + {self.params['R2']['r']}")
        print(f"     = {f2:.4f}")

        print(f"  f₃ = {self.params['R3']['p']}×{x1} + {self.params['R3']['q']}×{x2} + {self.params['R3']['r']}")
        print(f"     = {self.params['R3']['p']*x1:.4f} + {self.params['R3']['q']*x2:.4f} + {self.params['R3']['r']}")
        print(f"     = {f3:.4f}")

        print(f"  f₄ = {self.params['R4']['p']}×{x1} + {self.params['R4']['q']}×{x2} + {self.params['R4']['r']}")
        print(f"     = {self.params['R4']['p']*x1:.4f} + {self.params['R4']['q']*x2:.4f} + {self.params['R4']['r']}")
        print(f"     = {f4:.4f}")

        # Layer 5
        print("\n🎯 LAYER 5: AGREGASI (Weighted Average)")
        output = self.layer5_agregasi(w_bar1, w_bar2, w_bar3, w_bar4, f1, f2, f3, f4)
        print(f"  f = Σ(w̄ᵢ × fᵢ)")
        print(f"    = ({w_bar1:.4f} × {f1:.4f}) + ({w_bar2:.4f} × {f2:.4f})")
        print(f"      + ({w_bar3:.4f} × {f3:.4f}) + ({w_bar4:.4f} × {f4:.4f})")
        print(f"    = {w_bar1*f1:.4f} + {w_bar2*f2:.4f} + {w_bar3*f3:.4f} + {w_bar4*f4:.4f}")
        print(f"    = {output:.4f}")

        # Interpretasi
        if output < 3:
            kategori = "RENDAH"
            rekomendasi = "Tingkatkan irigasi dan pemupukan"
        elif output < 5:
            kategori = "SEDANG-BAIK"
            rekomendasi = "Pertahankan kelembaban, monitor pH 6-7, pemupukan NPK seimbang"
        else:
            kategori = "BAIK"
            rekomendasi = "Kondisi optimal, pertahankan praktik saat ini"

        print(f"\n✅ HASIL PREDIKSI:")
        print(f"  Produksi Jagung: {output:.2f} ton/ha")
        print(f"  Kategori: {kategori}")
        print(f"  Rekomendasi: {rekomendasi}")
        print("="*60 + "\n")

        return output, kategori


# =====================================================
# MAIN PROGRAM - KASUS B
# =====================================================

def main():
    print("LAPORAN UAS - FUZZY LOGIC & ANFIS")
    print("KASUS B: ANFIS (Adaptive Neuro-Fuzzy Inference System)")

    # Inisialisasi sistem ANFIS
    anfis = ANFIS()

    # Data input dari soal
    soil_moisture = 60  # x₁ = 60%
    soil_ph_temp = 26   # x₂ = 26

    # Proses prediksi
    produksi, kategori = anfis.prediksi(soil_moisture, soil_ph_temp)

    # Penjelasan Arsitektur ANFIS
    print("ARSITEKTUR ANFIS (5 LAYER)")
    print("Layer 1: Fuzzifikasi")
    print("  • Menggunakan fungsi Gaussian: μ(x) = exp(-((x-c)²)/(2σ²))")
    print("  • Mengkonversi input crisp ke derajat keanggotaan fuzzy")
    print("\nLayer 2: Firing Strength")
    print("  • Operasi AND (perkalian) antar derajat keanggotaan")
    print("  • Menghasilkan kekuatan aktivasi setiap rule")
    print("\nLayer 3: Normalisasi")
    print("  • Menormalisasi firing strength: w̄ᵢ = wᵢ / Σwᵢ")
    print("  • Memastikan total bobot = 1")
    print("\nLayer 4: Output Rule (Sugeno Orde-1)")
    print("  • fᵢ = pᵢ×x₁ + qᵢ×x₂ + rᵢ")
    print("  • Output linear terhadap input")
    print("\nLayer 5: Agregasi")
    print("  • Weighted average: f = Σ(w̄ᵢ × fᵢ)")
    print("  • Menghasilkan output crisp final")
    print("="*60 + "\n")

    # Kesimpulan
    print("KESIMPULAN")
    print("ANFIS menggabungkan kelebihan:")
    print("  • Fuzzy Logic: Kemampuan reasoning linguistik")
    print("  • Neural Network: Kemampuan learning dari data")
    print("\nKeunggulan ANFIS:")
    print("  ✓ Prediksi numerik presisi tinggi")
    print("  ✓ Adaptif (dapat training dengan data)")
    print("  ✓ Output crisp langsung (tanpa defuzzifikasi)")
    print("  ✓ Cocok untuk forecasting & prediksi")
    print("\nAplikasi:")
    print("  • Prediksi produksi pertanian")
    print("  • Forecasting ekonomi")
    print("  • Prediksi cuaca")
    print("  • Control system adaptif")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

LAPORAN UAS - FUZZY LOGIC & ANFIS
KASUS B: ANFIS (Adaptive Neuro-Fuzzy Inference System)

📊 DATA INPUT:
  Soil Moisture (x₁): 60%
  Soil pH/Temp (x₂): 26

🔍 LAYER 1: FUZZIFIKASI (Gaussian)
  μ_A1(60) = exp(-((60-40)²)/(2×15²)) = 0.4111
  μ_A2(60) = exp(-((60-70)²)/(2×15²)) = 0.8007
  μ_B1(26) = exp(-((26-20)²)/(2×8²)) = 0.7548
  μ_B2(26) = exp(-((26-30)²)/(2×8²)) = 0.8825

⚙️  LAYER 2: FIRING STRENGTH
  w₁ = μ_A1 × μ_B1 = 0.4111 × 0.7548 = 0.3103
  w₂ = μ_A1 × μ_B2 = 0.4111 × 0.8825 = 0.3628
  w₃ = μ_A2 × μ_B1 = 0.8007 × 0.7548 = 0.6044
  w₄ = μ_A2 × μ_B2 = 0.8007 × 0.8825 = 0.7066
  Total = 1.9842

🔄 LAYER 3: NORMALISASI
  w̄₁ = 0.3103 / 1.9842 = 0.1564
  w̄₂ = 0.3628 / 1.9842 = 0.1828
  w̄₃ = 0.6044 / 1.9842 = 0.3046
  w̄₄ = 0.7066 / 1.9842 = 0.3561
  Verifikasi: Σw̄ = 1.0000 ≈ 1.0 ✓

📐 LAYER 4: OUTPUT TIAP RULE (Sugeno Orde-1)
  f₁ = 0.0692×60 + 0.0461×26 + 0.00231
     = 4.1520 + 1.1986 + 0.00231
     = 5.3529
  f₂ = 0.0698×60 + 0.039×26 + 0.00139
     = 4.1880 + 1.0140 + 0.00139
 